In [18]:
import json
import math
import random

import torch

import pandas as pd
import os

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm


import matplotlib.pyplot as plt
import pandas as pd
import itertools
import time
from statistics import mode
from transformers import logging

logging.set_verbosity_error()



In [19]:
config = {
   "MODEL_ID":  "Qwen/Qwen2.5-3B-Instruct",#"Qwen/Qwen2.5-3B-Instruct" ,#"microsoft/phi-2",
   "DATASET_ID" : "qasc", ##"medmcqa","teleqna", "qasc" 
   "DATASET_PATH" :  "/home/ubuntu/qmos/data/telcom_llm/QASC_Dataset_1Step/dev.jsonl"  ,#"/home/ubuntu/qmos/data/telcom_llm/medmcqa/dev.json", #"/home/ubuntu/qmos/data/questions_new_final_backup.json"

}


In [21]:
# Function to clean and split text into words
def preprocess(text):
    
    text = text.lower()
    return set(text.split())


def choose_most_likely_option(options, candidate_answer):
    candidate_words = preprocess(candidate_answer)
    
    best_option = None
    max_overlap = 0
    
    for option in options:
        option_words = preprocess(option)
        overlap = len(candidate_words.intersection(option_words))
        
  
        if overlap > max_overlap:
            max_overlap = overlap
            best_option = option
            
    return best_option, options.index(best_option)+1

In [22]:
from string import Template

## Choose model ID

MODEL_ID = config["MODEL_ID"]
DATASET_ID = config["DATASET_ID"]
DATASET_PATH = config["DATASET_PATH"]

if  DATASET_ID == "medmcqa" and MODEL_ID in ["Qwen/Qwen2.5-3B-Instruct" ,"microsoft/phi-2"]:
    prompt_q_without_contex_train= Template('''Instruct = Youre a Medical Question Answering Expert, answer the following question. Please generate only answer choice (1, 2, 3 or 4)\n
    $question
    $options
    $question
    ''')


    prompt_without_contex_train= Template('''Instruct = Youre a Medical Question Answering Expert, answer the following question. Please generate only answer choice (1, 2, 3 or 4)\n                                                                 
    $question
    $options
    Output: option ''')




    cache_prompt= Template('''Instruct : Youre a Medical Question Answering Expert, answer the following question. Please generate only answer choice (1, 2, 3 or 4)\n                                                  
    $question
    ''')

    option_prompt = Template('''
    $options
    Output: option ''')
    # prompt_without_context = f'Hello {planet}'

elif DATASET_ID == "teleqna" and MODEL_ID == "Qwen/Qwen2.5-3B-Instruct" :


    cache_prompt= Template('''Instruct: $question
    Abbreviations: $abbreviation
            
    Considering the following contexts:
    context 1: $context1
    context 2: $context2
    context 3: $context3      
                                                                        
    $question
    ''')    

    option_prompt = Template('''
    $options
    Output: option ''')
    
    prompt_without_contex_train= Template('''Instruct: $question
    Abbreviations: $abbreviation
            
    Considering the following contexts:
    context 1: $context1
    context 2: $context2
    context 3: $context3      
                                                                        
    $question
    $options
    Output: option ''')

elif DATASET_ID == "teleqna" and MODEL_ID == "microsoft/phi-2":


    cache_prompt= Template('''Instruct: Answer the following question using the context provided.Your answer must start with the correct option letter (A, B, C, D, or E) followed by the text of the answer: $question
    Abbreviations: $abbreviation
            
    Considering the following contexts:
    context 1: $context1
    context 2: $context2
    context 3: $context3      
                                                                        
    $question
    ''')    

    option_prompt = Template('''
    $options
    Output: option ''')


    prompt_without_contex_train= Template('''Instruct: $question
    Abbreviations: $abbreviation
            
    Considering the following contexts:
    context 1: $context1
    context 2: $context2
    context 3: $context3      
                                                                        
    $question
    $options
    Output: option ''')

elif DATASET_ID == "qasc" and MODEL_ID in ["Qwen/Qwen2.5-3B-Instruct" ,"microsoft/phi-2"]:

    prompt_without_contex_train = Template('''Instruct: Answer the following question using the context provided, reason over it because only one of the context is relevant . Please generate only answer choice (1, 2, 3, 4, 5, 6, 7 or 8) without any explanations\n
    $question
    context: $context      
                                                            
    $options
    $question
    Output: option 
    ''')

    cache_prompt= Template('''Instruct: Answer the following question using the context provided, reason over it because only one of the context is relevant . Please generate only answer choice (1, 2, 3, 4, 5, 6, 7 or 8) without any explanations\n
    $question
    context: $context      
    ''')

    option_prompt = Template('''
    $options
    Output: option ''')


In [23]:
def clean_question(question):
    for num in [14, 15, 16, 17, 18]:
        question = question.replace(f"[3GPP Release {num}]", "")
    return question

In [24]:
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "auto"
# torch.set_default_device(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True, device_map="auto")
# model.to(device)



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [25]:
class MyLLMDataloader:
    def __init__(self, batch_size, tokenizer, data, shuffle = False, val= False, k=20):
        ## initializations
        self.batch_size  = batch_size
        self.tokenizer  = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        # with open(data, "r") as f:
        #     self.data = json.load(f)
        if DATASET_ID == "medmcqa" or DATASET_ID == "qasc":
            self.data = []
            with open(data, "r") as f:
                test_data = f.readlines()

            for line in test_data:
                self.data.append(json.loads(line))
        elif DATASET_ID == "teleqna" :
            with open(data, "r") as f:
                self.data = json.load(f)
            self.all_examples = list(self.data.keys())

            
        # self.all_examples = list(self.data.keys())
        self.shuffle = shuffle
        self.val = val
        self.k = k
        self.n_data_points = math.ceil(len(self.data)/self.batch_size)
        self.indices = [i for i in range(self.n_data_points)]
        
    def __getitem__(self, idx):
        ## this gets a batch 

        option_header = ["option 1 ", "option 2 ", "option 3 ", "option 4 ", "option 5 "]
        batch_start_id = idx * self.batch_size
        mapper_ans = {"a":1, "b":2, "c":3, "d":4, "e":5}
        batch_end_id  = min(len(self.data), batch_start_id + self.batch_size) 
        batch = {"options":[], "answer":[], "question_context":[],}
        batch_type =False
        
        if DATASET_ID == "medmcqa":
            for i in range(batch_start_id, batch_end_id):
                example = self.data[i]
                options = []
                opts = []
                for key in example.keys():
                    if key.startswith("op"):
                        if example[key] == None:
                            continue
                        options.append(example[key])
                        opts.append((example[key], mapper_ans[key.split("op")[1]] ))

        elif DATASET_ID == "teleqna":
            for i in range(batch_start_id, batch_end_id):
                example = self.data[self.all_examples[i]]
                options = []
                opts = []
                for key in example.keys():
                    if key.startswith("opt"):
                        if example[key] == None:
                            continue
                        options.append(example[key])
                        opts.append((example[key], key.split("option ")[1]))
            
        elif DATASET_ID == "qasc":
                mapper_ans = {"A":1, "B":2, "C":3, "D":4, "E":5, "F":6, "G":7, "H":8}
                option_header = ["option 1 ", "option 2 ", "option 3 ", "option 4 ", "option 5 ","option 6 ", "option 7 ", "option 8 " ]

                for i in range(batch_start_id, batch_end_id):
                    example = self.data[i]
                    options = []
                    opts = []
                    context = example["fact1"] + '\n' + example["fact2"]
                    for i, sample in enumerate(example['question']['choices']):
                        # print(key)
                        if example["answerKey"] == sample["label"]:
                            correct_option_id = mapper_ans[example["answerKey"]]
                            correct_option_txt = sample["text"]
                        
                            
                    
                        # options.append(option_header[i]+ sample['text'])
                        options.append(sample['text'])

                        opts.append((sample['text'], option_header[i].split("option")[1]))
                 
                explanation = example['combinedfact']

        string_opts = ' '.join(opt[0] for opt in opts)
        batch_prompts = []
        option_maps = []
        if not ("option" in string_opts or "above" in string_opts) and self.k>1:
                batch_type = True
                if DATASET_ID == "medmcqa":

                    question_context = cache_prompt.substitute(question = clean_question(example["question"]))
                elif DATASET_ID == "teleqna":
                    question_context = cache_prompt.substitute(question = clean_question(example["question"]),\
                    abbreviation='\n'.join(example["abbreviation"]), context1 = '\n'.join(example["context_qwen2"][:2]) , context2 = '\n'.join(example["context_gle"]), context3 = '\n'.join(example["context_bm"][:2]))

                elif DATASET_ID == "qasc":
                    question_context = cache_prompt.substitute(question = clean_question(correct_option_txt), context = explanation)

                batch["question_context"].append(question_context)

                all_permutations = list(itertools.permutations(opts))
                all_permutations = random.sample(all_permutations, self.k if len(all_permutations)>self.k else len(all_permutations))
                
                for option_set in all_permutations:
                    option_map = []
                    options_with_header   = []
                    for z in range(len(option_set)):

                        options_with_header.append(option_header[z] +option_set[z][0])

                        
                        option_map.append(int(option_set[z][1]))
                    
                    
                    
                    options_with_header = "\n".join(options_with_header)

                    prompt= option_prompt.substitute(options =options_with_header)
                    # print(question_context, prompt)
                    batch_prompts.append(prompt)
                    option_maps.append(option_map)
                batch["options"] += batch_prompts

                    


        else:
            if DATASET_ID == "medmcqa":
            
                options_with_header = [option_header[i] +options[i] for i in range(len(options)) ]
            
                options_with_header = "\n".join(options_with_header)
                prompt = prompt_without_contex_train.substitute(question = clean_question(example["question"]), options =options_with_header)
            elif DATASET_ID == "teleqna":
                options_with_header = [option_header[i] +options[i] for i in range(len(options)) ]
                options_with_header = "\n".join(options_with_header)
                prompt = prompt_without_contex_train.substitute(question = clean_question(example["question"]),\
                abbreviation='\n'.join(example["abbreviation"]), context1 = '\n'.join(example["context_qwen2"][:2]) , context2 = '\n'.join(example["context_gle"]), context3 = '\n'.join(example["context_bm"][:2]),
                options =options_with_header)     
            
            elif DATASET_ID == "qasc":
                correct_option_id = correct_option_id
                correct_option_txt_header = str(correct_option_id) + " " + correct_option_txt
                options_with_header = [option_header[i] +options[i] for i in range(len(options)) ]
                options_with_header = "\n".join(options_with_header)
                prompt = prompt_without_contex_train.substitute( question  = example['question']['stem'], context=  context, options = options_with_header)+ "option"

                

            batch_prompts.append(prompt)
            

            


            batch["options"] += batch_prompts
            # batch["answer"] += [answer]
        self.tokenizer.padding_side = "left"
        question_context_tokens =  {"input_ids":None,"attention_mask":None}
        q_tokens = self.tokenizer(batch["options"], padding="longest", return_tensors="pt")  

        if batch_type:
            # print("orig mask", q_tokens["attention_mask"].shape, q_tokens["attention_mask"])
            question_context_tokens = self.tokenizer(batch["question_context"], padding="longest", return_tensors="pt")  
            attn_masks = torch.ones((q_tokens["attention_mask"].shape[0], q_tokens["attention_mask"].shape[1] + len(question_context_tokens[0])))
        else:
            attn_masks = q_tokens["attention_mask"]
        self.tokenizer.padding_side = "right"
        # a_tokens = self.tokenizer(batch["answer"], padding="longest", return_tensors="pt")
        tokens = q_tokens
        
       

        # attn_masks = torch.cat([q_tokens["attention_mask"], a_tokens["attention_mask"]], dim=1)
        # loss_mask = torch.cat([torch.zeros_like(q_tokens["attention_mask"]), a_tokens["attention_mask"]], dim=1)[:,1:]
   
        result = {
        "inp_ids":tokens["input_ids"],
        "inp_mask":attn_masks,## Causal Training
        "option_maps": option_maps,
        "cached": batch_type,
        "question_context_ids":question_context_tokens["input_ids"],
        "question_mask":question_context_tokens["attention_mask"],


        }

        # result["loss_mask"] = loss_mask * result["out_mask"]
        # result["out_ids"][:,:q_tokens["input_ids"].size(1)-10] = self.tokenizer.eos_token_id

        return result       


            

    def __iter__(self):
        self.idx = 0
        return self

    def __next__(self):
        if self.idx >= self.n_data_points:
            self.idx = 0
            raise StopIteration
        temp_idx = self.indices[self.idx]
        self.idx += 1
        return self[temp_idx]
             





    def __len__(self):
        return self.n_data_points
    



        

In [26]:
k=20
testLoader = MyLLMDataloader(1, tokenizer, DATASET_PATH, val=True, shuffle=False, k=k)
for doc in testLoader:
    print(doc.keys())

    break

dict_keys(['inp_ids', 'inp_mask', 'option_maps', 'cached', 'question_context_ids', 'question_mask'])


In [27]:
DATASET_ID


'qasc'

In [28]:
def forward_pass(model, batch, k=5):
    inp_ids = batch["inp_ids"].to(model.device)
    attn_mask = batch["inp_mask"].to(model.device)

    if batch["cached"]:
        with torch.inference_mode():
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                prompt_cache = model(input_ids=batch["question_context_ids"], attention_mask=batch["question_mask"], use_cache = True).past_key_values
                # Convert prompt_cache to a list if it's a tuple
                prompt_cache = list(prompt_cache)
                qc_len = prompt_cache[0][0].shape[2]
                with open('tokenssavings'+DATASET_ID+"_"+ MODEL_ID.split('/')[1]+ '.txt', 'a') as the_file:
                    the_file.write(str((k-1)*qc_len/ (k *attn_mask.shape[1])) +'\n')

              
                for i, (ke, v) in enumerate(prompt_cache):
                
                    k_repeated = ke.repeat(k, 1, 1, 1)
                    # print(ke.shape, k_repeated.shape)
                    v_repeated = v.repeat(k, 1, 1, 1)

        
                    prompt_cache[i] = (k_repeated, v_repeated)

        
                prompt_cache = tuple(prompt_cache)



                
                result = model(input_ids=inp_ids, attention_mask=attn_mask, past_key_values=prompt_cache)
        return result.logits

    result = model(input_ids=inp_ids, attention_mask=attn_mask)
    logits = result.logits
    return logits

In [29]:
def inference(model, testLoader, k = 15):
    my_ans = {"Answer_ID": []}
 
    model.eval()          
    option_ids = [tokenizer(o).input_ids[0] for o in ["1", "2", "3", "4", "5"]]
    pbar = tqdm(range(len(testLoader)), )
    for item in testLoader:
        
        # if int(tokenizer.decode(item["a_tokens"].input_ids[:,0], skip_special_tokens=True)) ==0:
        if len(item["option_maps"]) >0:
            # print('\n'.join(tokenizer.batch_decode(item['inp_ids']))),
            # first_half = item.copy()

        




            # print(first_half["inp_ids"].shape, second_half["inp_ids"].shape)
            with torch.inference_mode():
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    # gen_tokens = model.generate(inputs=item["inp_ids"].to(device), max_new_tokens=1)

                    start = time.time()
                    logits = forward_pass(model, item,k=item["inp_ids"].shape[0])
                    with open('time_bqkv'+DATASET_ID+"_"+ MODEL_ID.split('/')[1]+'.txt', 'a') as the_file:
                            the_file.write(str(time.time()- start) +'\n')
                        # logits2 = forward_pass(model, second_half)
                  
                
            
  
            # print(logits.shape)
            
            preds =(logits[:, -1, option_ids ].argmax(dim=1) ).cpu()
            z = torch.tensor(item["option_maps"])
            # print("predictions",preds, z)
         
            # print( tokenizer.batch_decode(item["inp_ids"]),"predicted", preds.item(), tokenizer.batch_decode(logits.argmax(axis=2), skip_special_tokens=True)[0])
            preds = torch.mode(z.gather(1, preds.unsqueeze(1)).squeeze(1)).values
            my_ans["Answer_ID"].append(preds.item())
            # print("logits", tokenizer.batch_decode(gen_tokens,  skip_special_tokens=True)[0] )
            # print(tokenizer.decode(item["a_tokens"].input_ids[:,0], skip_special_tokens=True))
            # break
        else:
            with torch.inference_mode():
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    # gen_tokens = model.generate(inputs=item["inp_ids"].to(device), max_new_tokens=1)
                    start = time.time()
                    logits = forward_pass(model, item, k =1)
                    with open('time_bqkv'+ DATASET_ID+"_"+ MODEL_ID.split('/')[1]+'.txt', 'a') as the_file:
                        the_file.write(str(time.time()- start) +'\n')
                    preds =(logits[:, -1, option_ids ].argmax(dim=1) +1)


                    my_ans["Answer_ID"].append(preds.item())
        
        pd.DataFrame(my_ans).to_csv("medmcq_ans"+str(k)+".csv")
        pbar.set_description(f"Prediction: {preds}")
        pbar.update(1)
    return my_ans

    

In [30]:
# k = 15
# testLoader = MyLLMDataloader(1, tokenizer, "questions_new_final_backup.json", val=True, shuffle=False, k=k)
# model.eval() 
# i = 0
# for dat in testLoader:
#     if i ==1:
#         start = time.time()
    
#         logit = forward_pass(model, dat,k=k)
#         print(time.time()-start)

#         break
#     i+=1

In [31]:
k=1
testLoader = MyLLMDataloader(1, tokenizer, DATASET_PATH, val=True, shuffle=False, k=k)
for doc in testLoader:
    print(doc.keys())
    break

dict_keys(['inp_ids', 'inp_mask', 'option_maps', 'cached', 'question_context_ids', 'question_mask'])


In [32]:
k = 20
testLoader = MyLLMDataloader(1, tokenizer, DATASET_PATH, val=True, shuffle=False, k=k)


predictions = inference(model, testLoader,  k =k)

pd.DataFrame(predictions).to_csv(DATASET_ID+"_" + MODEL_ID.split("/")[1] +"_k_"+ str(k)+".csv")

  0%|          | 0/926 [00:00<?, ?it/s]

In [1]:
with open("time_bqkvqasc_Qwen2.5-3B-Instruct.txt", "r") as f:
    file = f.readlines()

total_time = 0
for line in file:
    
    total_time += float(line.splitlines()[0])
print("average time ", total_time/len(file))


with open("tokenssavingsmedmcqa_phi-2.txt", "r") as f:
    file = f.readlines()

total_time = 0
for line in file:
    
    total_time += float(line.splitlines()[0])
print("average tokens saved ", total_time/len(file))


average time  0.1235616915169339
average tokens saved  0.5640061228241096
